In [ ]:
from datetime import datetime
from pathlib import Path

import polars as pl

## Process UCI Dataset

This notebook preprocesses the UCI electricity load dataset by removing low-quality client sites, filtering problematic time ranges, and interpolating missing timesteps to create a clean long-format dataset.

In [ ]:
from constants import UCI_CLIENTS_TO_DROP, UCI_CLIENTS_TO_FILTER, UCI_CLIENTS_TO_INTERPOLATE

In [ ]:
FREQUENCY_MINUTES = 15

UCI_DATA_DIR = Path("../../data/uci")
UCI_DATA_LOAD_PATH = UCI_DATA_DIR / "LD2011_2014.txt"
UCI_DATA_SAVE_PATH = UCI_DATA_DIR / "preprocessed.pq"

In [ ]:
# Load data as polars dataframe

UCI_DF = pl.read_csv(
    UCI_DATA_LOAD_PATH,
    has_header=True,
    separator=";",
    decimal_comma=True,
    try_parse_dates=True,
    infer_schema_length=1_000_000
)

# First column should be timestamp column
UCI_DF = UCI_DF.rename({UCI_DF.columns[0]: "timestamp"}).sort(by="timestamp")

In [ ]:
# Drop client sites with poor data quality

def drop_client_timeseries(uci_df: pl.DataFrame, client_name: str) -> pl.DataFrame:
    """
    Remove a client sites timeseries column from the dataframe.
    """
    return uci_df.select(pl.all().exclude(client_name))


for client in UCI_CLIENTS_TO_DROP:
    UCI_DF = drop_client_timeseries(uci_df=UCI_DF, client_name=client)

In [ ]:
# Filter client timeseries to timerange with good quality data

def filter_client_timeseries(
    uci_df: pl.DataFrame,
    client_name: str,
    start_ts: datetime,
    end_ts: datetime,
) -> pl.DataFrame:
    """
    Filter a client site's timeseries to a specific time range and set values outside the range to zero.
    """

    client_df = (
        uci_df
        .select(pl.col("timestamp"), pl.col(client_name))
        .filter(pl.col("timestamp").is_between(start_ts, end_ts, closed="both"))
    )

    # Merge back onto main df
    uci_df = (
        uci_df.select(pl.all().exclude(client_name))
        .join(client_df, on="timestamp", how="left")
        .select(pl.all().exclude(client_name), pl.col(client_name).fill_null(0.0))
    )

    # Sort columns
    uci_df = uci_df.select(
        pl.col("timestamp"),
        *[pl.col(c) for c in sorted([c for c in uci_df.columns if c != "timestamp"])]
    )

    return uci_df


for client_name, (start_ts, end_ts) in UCI_CLIENTS_TO_FILTER:
    UCI_DF = filter_client_timeseries(
        uci_df=UCI_DF,
        client_name=client_name,
        start_ts=start_ts,
        end_ts=end_ts
    )

In [ ]:
# Interpolate missing timesteps

def get_min_max_timestamps_by_client(uci_df: pl.DataFrame) -> pl.DataFrame:
    """
    Compute the minimum and maximum timestamps where a client site has non-zero values.
    """
    min_max_ts = (
        uci_df.unpivot(
            on=[c for c in uci_df.columns if c != "timestamp"],
            index="timestamp",
            variable_name="client",
        )
        .filter(pl.col("value") > 0)
        .group_by("client", maintain_order=True)
        .agg(
            min_timestamp=pl.col("timestamp").min(),
            max_timestamp=pl.col("timestamp").max(),
        )
    )
    return min_max_ts


def interpolate_client_timeseries(
    uci_df: pl.DataFrame,
    client_name: str,
    start_ts: datetime,
    end_ts: datetime,
    interval: str = "15m",
) -> pl.DataFrame:
    """
    Interpolate missing timesteps for a client's timeseries within a specified time range
    """

    # Get all non-zero observations between start_ts and end_ts
    client_df = uci_df.select(pl.col("timestamp"), pl.col(client_name)).filter(
        pl.col("timestamp").is_between(start_ts, end_ts, closed="both"),
        pl.col(client_name) > 0,
    )

    # Construct timeseries of expected timestamps between start_ts and end_ts
    expected_ts = pl.datetime_range(
        start=start_ts,
        end=end_ts,
        interval=interval,
        closed="both",
        eager=True,
    )

    # Merge and interpoalte
    client_df = (
        expected_ts.to_frame(name="timestamp")
        .join(client_df, on="timestamp", how="left")
        .select(pl.col("timestamp"), pl.col(client_name).interpolate())
    )

    # Merge back onto original uci_df
    uci_df = (
        uci_df.select(pl.all().exclude(client_name))
        .join(client_df, on="timestamp", how="left")
        .select(pl.all().exclude(client_name), pl.col(client_name).fill_null(0.0))
    )

    # Sort columns
    uci_df = uci_df.select(
        pl.col("timestamp"),
        *[pl.col(c) for c in sorted([c for c in uci_df.columns if c != "timestamp"])],
    )

    return uci_df



# Interpolate client timeseries
min_max_timestamp_by_client = get_min_max_timestamps_by_client(UCI_DF)
for client_name in UCI_CLIENTS_TO_INTERPOLATE:
    # Get min / max timestamps for this client
    client_min_max_ts = min_max_timestamp_by_client.filter(pl.col("client") == client_name)
    [client_min_ts] = client_min_max_ts["min_timestamp"].to_list()
    [client_max_ts] = client_min_max_ts["max_timestamp"].to_list()

    UCI_DF = interpolate_client_timeseries(
        uci_df=UCI_DF,
        client_name=client_name,
        start_ts=client_min_ts,
        end_ts=client_max_ts,
        interval=f"{FREQUENCY_MINUTES}m"
    )

In [ ]:
# Unpivot UCI DF

long_sample_clients_demand_table = (
    UCI_DF
    # Unpivot separate client columns into single column
    # with client name as variable
    .unpivot(
        on=[c for c in UCI_DF.columns if c != "timestamp"],
        index="timestamp",
        variable_name="client",
        value_name="demand"
    )
    # Add the min / max timestamps for each client and timestamp record
    .join(
        other=min_max_timestamp_by_client,
        on="client",
        how="left",
    )
    # Add a column for each timestamp indicating if that timestamp record is
    # in range i.e. within the client's min / max timestamps
    .with_columns(in_range=pl.col("timestamp").is_between("min_timestamp", "max_timestamp"))
    # Filter to only keep records that are in range.
    .filter(pl.col("in_range"))
    # Only keep timestamp, client and demand columns
    .select(pl.col("timestamp"), pl.col("client"), pl.col("demand"))
)

In [ ]:
# Save

long_sample_clients_demand_table.to_pandas().to_parquet(UCI_DATA_SAVE_PATH)